In [ ]:
# Pagination example

In [84]:
import requests
import json
import time
from datetime import datetime
import logging

URL = "https://rickandmortyapi.com/api/character"
OUTPUT_FILE = "../output/rick_morty.jsonl"

logging.basicConfig(level=logging.INFO)

In [85]:
def extract_character(start_url):
    next_url = start_url
    page_count = 1
    max_page_count = 10

    while next_url and page_count <= max_page_count:
        logging.info(f"Extracting page {page_count}: {next_url}")
        try:
            response = requests.get(next_url, timeout=(5,15))
            response.raise_for_status()

            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After"))
                logging.info(f"Rate limited, sleeping for {retry_after} seconds")
                time.sleep(retry_after)
                continue

            payload = response.json()
        except json.JSONDecodeError:
            logging.error(f"Failed to parse JSON on page {page_count}")
            break
        except Exception as e:
            logging.error(f"Exception: {e}")
            break

        characters = payload.get('results',[])
        for char in characters:
            yield char
        
        # if I'm just adding all the data to a list instead of yielding a single character
        # all_data.extend(data['results'])
        #    page += 1

        next_url = payload.get('info', {}).get('next')
        page_count += 1

        time.sleep(1)

In [ ]:
def transform_character(char):
    curr_time = datetime.now()
    return {
        "character_id": char.get("id"),
        "name": char.get("name"),
        "gender": char.get("gender"),
        "origin_name": char.get("origin").get("name"),
        "extracted_at": curr_time.strftime("%Y-%m-%dT%H:%M:%SZ")
    }

In [87]:
def run_pipeline():
    logging.info("Starting ETL pipeline")
    record_count = 0
    with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
        for record in extract_character(URL):
            transformed_record = transform_character(record)

            file.write(json.dumps(transformed_record) + "\n")
            record_count += 1
    
    logging.info(f"Pipeline complete, wrote {record_count} records")

In [88]:
run_pipeline()

INFO:root:Starting ETL pipeline
INFO:root:Extracting page 1: https://rickandmortyapi.com/api/character
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:___ Transforming record ___
INFO:root:Extracting page 2: https://rickandmortyapi.com/api/character?page=2
INFO:root:___ Transforming record ___
INFO:root:___ Transfo